In [11]:
import pandas as pd
import json
import os
import deepsig
from IPython.display import display

In [12]:
cols = ['dataset', 'method', 'fitness_rule', 'fitness', 'ACC', 'MCC', 'f1_score', 'avg_odds_diff', 'stat_par_diff', 'eq_opp_diff']

In [13]:
mlp_baseline_results = pd.read_csv('simple_mlp_results.csv')
mlp_baseline_results.replace({'simple_mlp_initializer': r'MLP'}, inplace=True)

mlp_standard_l2_results = pd.read_csv('mlp_standard_l2_results.csv')
mlp_standard_l2_results.replace({'mlp_standard_l2_initializer': r'MLP+L2'}, inplace=True)

mlp_featurewise_l2_results = pd.read_csv('mlp_featurewise_l2_results.csv')
mlp_featurewise_l2_results.replace({'mlp_featurewise_l2_initializer': r'MLP+L2^{(1)}'}, inplace=True)

mlp_preg_results = pd.read_csv('mlp_preg_results.csv')
mlp_preg_results.replace({'mlp_preg_initializer': r'MLP+SDR$_{\rho}$'}, inplace=True)

mlp_sreg_results = pd.read_csv('mlp_sreg_results.csv')
mlp_sreg_results.replace({'mlp_sreg_initializer': r'MLP+SDR$_{\rho_s}$'}, inplace=True)

mlp_kreg_results = pd.read_csv('mlp_kreg_results.csv')
mlp_kreg_results.replace({'mlp_kreg_initializer': r'MLP+SDR$_{\tau}$'}, inplace=True)

mlp_xi_reg_results = pd.read_csv('mlp_xi_reg_results.csv')
mlp_xi_reg_results.replace({'mlp_xi_reg_initializer': r'MLP+SDR$_{\xi}$'}, inplace=True)

results = pd.concat([mlp_baseline_results, mlp_standard_l2_results, mlp_featurewise_l2_results, 
                     mlp_preg_results, mlp_sreg_results, mlp_kreg_results, mlp_xi_reg_results], 
                    ignore_index=True)

In [14]:
results.replace({'adult_dataset_reader': 'Adult Income', 
                 'compas_dataset_reader': 'Compas Recidivism', 
                 'german_dataset_reader': 'German Credit', 
                 'bank_dataset_reader': 'Bank Marketing'}, inplace=True)

results.rename(columns={'avg_odds_diff': 'Equalized Odds', 
                        'stat_par_diff': 'Statistical Parity', 
                        'eq_opp_diff': 'Equal Opportunity', 
                        'MCC': 'Mathew Correlation', 
                        'ACC': 'Accuracy'}, inplace=True)

In [15]:
fitness_rules_target_metrics = {
    'mcc_parity': ('Mathew Correlation', 'Statistical Parity'),
    'mcc_opportunity': ('Mathew Correlation', 'Equal Opportunity'),
    'mcc_odds': ('Mathew Correlation', 'Equalized Odds'),
    'acc_parity': ('Accuracy', 'Statistical Parity'),
    'acc_opportunity': ('Accuracy', 'Equal Opportunity'),
    'acc_odds': ('Accuracy', 'Equalized Odds')
}

fitness_rules_abvr = {
    'mcc_parity': 'Max(MCC - Stat. Parity)',
    'mcc_opportunity': 'Max(MCC - Eq. Odds)',
    'mcc_odds': 'Max(MCC - Eq. Opp.)',
    'acc_parity': 'Max(Acc - Stat. Parity)',
    'acc_opportunity': 'Max(Acc - Eq. Odds)',
    'acc_odds': 'Max(Acc - Eq. Opp.)'
}

results['Performance'] = 0.0
results['Fairness'] = 0.0
results['Fitness Rule'] = ''

for fitness_rule, (performance_metric, fairness_metric) in fitness_rules_target_metrics.items():
    mask = results.fitness_rule == fitness_rule
    results.loc[mask, 'Performance'] = results.loc[mask, performance_metric]
    results.loc[mask, 'Fairness'] = results.loc[mask, fairness_metric]
    results.loc[mask, 'Fitness Rule Abvr'] = fitness_rules_abvr[fitness_rule]
    results.loc[mask, 'Fitness Rule'] = 'Max(%s - %s)' % fitness_rules_target_metrics[fitness_rule]

In [16]:
datasets = ['Adult Income', 'Bank Marketing', 'Compas Recidivism', 'German Credit']
fitness_rules = ['mcc_parity', 'mcc_opportunity', 'mcc_odds', 'acc_parity', 'acc_opportunity', 'acc_odds']

mlp_methods = [r'MLP', r'MLP+L2', r'MLP+L2^{(1)}', r'MLP+SDR$_{\rho}$', 
               r'MLP+SDR$_{\rho_s}$', r'MLP+SDR$_{\tau}$', r'MLP+SDR$_{\xi}$']

## MultiASO Analysis: MLP+SDR$_{\xi}$ vs Other MLP Methods

Comparação usando MultiASO para testar MLP+SDR$_{\xi}$ contra todos os outros métodos MLP simultaneamente.

In [17]:
multi_aso_path = 'mlp_xi_multi_aso_results.json'

if os.path.exists(multi_aso_path):
    with open(multi_aso_path) as file:
        multi_aso_data_list = json.load(file)
else:
    multi_aso_data_list = []
    
    for d in datasets:
        for f in fitness_rules:
            # Coletar resultados de todos os métodos MLP para este dataset/fitness_rule
            all_samples = []
            method_names = []
            
            for method in mlp_methods:
                samples = results.loc[(results['dataset'] == d) &
                                     (results['fitness_rule'] == f) &
                                     (results['method'] == method)].fitness.tolist()
                
                if len(samples) > 0:
                    all_samples.append(samples)
                    method_names.append(method)
            
            if len(all_samples) < 2:
                print(f"Skipping {d} - {f}: insufficient methods")
                continue
            
            print(f"\n{d} - {f}")
            print(f"Methods: {method_names}")
            print(f"Sample sizes: {[len(s) for s in all_samples]}")
            
            # Executar MultiASO
            try:
                min_eps_matrix = deepsig.multi_aso(all_samples, confidence_level=0.95)
                
                # Encontrar índice do MLP+SDR$_{\xi}$
                xi_idx = method_names.index(r'MLP+SDR$_{\xi}$')
                
                # Extrair comparações relevantes
                comparisons = {}
                for i, method in enumerate(method_names):
                    if i != xi_idx:
                        # ASO(MLP+SDR_xi, method) - xi é melhor
                        eps_forward = min_eps_matrix[xi_idx][i]
                        # ASO(method, MLP+SDR_xi) - method é melhor
                        eps_backward = min_eps_matrix[i][xi_idx]
                        
                        comparisons[method] = {
                            'eps_forward': eps_forward,
                            'eps_backward': eps_backward
                        }
                        
                        # Classificação
                        if eps_forward < 0.2:
                            interpretation = "significantly_better"
                        elif eps_forward < 0.5:
                            interpretation = "better"
                        elif eps_backward < 0.2:
                            interpretation = "significantly_worse"
                        elif eps_backward < 0.5:
                            interpretation = "worse"
                        else:
                            interpretation = "tie"
                        
                        comparisons[method]['interpretation'] = interpretation
                
                multi_aso_data_list.append({
                    'dataset': d,
                    'fitness_rule': f,
                    'comparisons': comparisons
                })
                
                print(f"MultiASO completed successfully")
                
            except Exception as e:
                print(f"Error in MultiASO: {e}")
                continue
    
    with open(multi_aso_path, 'w') as file:
        json.dump(multi_aso_data_list, file, indent=2)

## Visualização dos Resultados MultiASO

In [18]:
# Criar tabelas de resultados por fitness_rule
for fitness_rule in fitness_rules:
    print(f"\n{'='*80}")
    print(f"Fitness Rule: {fitness_rules_abvr[fitness_rule]}")
    print(f"{'='*80}\n")
    
    # Criar DataFrame para esta fitness_rule
    rows = []
    
    for entry in multi_aso_data_list:
        if entry['fitness_rule'] == fitness_rule:
            dataset = entry['dataset']
            
            for method, comp_data in entry['comparisons'].items():
                rows.append({
                    'Dataset': dataset,
                    'Method': method,
                    'ε(Xi→Method)': f"{comp_data['eps_forward']:.3f}",
                    'ε(Method→Xi)': f"{comp_data['eps_backward']:.3f}",
                    'Result': comp_data['interpretation']
                })
    
    if rows:
        df = pd.DataFrame(rows)
        display(df)
        
        # Salvar em LaTeX
        df.to_latex(f'tables/multi_aso_{fitness_rule}.tex', index=False)
        print(f"\nSaved to tables/multi_aso_{fitness_rule}.tex")


Fitness Rule: Max(MCC - Stat. Parity)



,Dataset,Method,ε(Xi→Method),ε(Method→Xi),Result
0,Adult Income,MLP,0.778,1.000,tie
1,Adult Income,MLP+L2,0.692,1.000,tie
2,Adult Income,MLP+L2^{(1)},1.000,0.174,significantly_worse
3,Adult Income,MLP+SDR$_{\rho}$,1.000,0.314,worse
4,Adult Income,MLP+SDR$_{\rho_s}$,0.611,1.000,tie
5,Adult Income,MLP+SDR$_{\tau}$,0.650,1.000,tie
6,Bank Marketing,MLP,1.000,1.000,tie
7,Bank Marketing,MLP+L2,0.466,1.000,better
8,Bank Marketing,MLP+L2^{(1)},0.009,0.999,significantly_better
9,Bank Marketing,MLP+SDR$_{\rho}$,0.777,1.000,tie



Saved to tables/multi_aso_mcc_parity.tex

Fitness Rule: Max(MCC - Eq. Odds)



,Dataset,Method,ε(Xi→Method),ε(Method→Xi),Result
0,Adult Income,MLP,0.309,1.000,better
1,Adult Income,MLP+L2,0.271,1.000,better
2,Adult Income,MLP+L2^{(1)},0.278,1.000,better
3,Adult Income,MLP+SDR$_{\rho}$,0.249,1.000,better
4,Adult Income,MLP+SDR$_{\rho_s}$,0.615,1.000,tie
5,Adult Income,MLP+SDR$_{\tau}$,0.453,1.000,better
6,Bank Marketing,MLP,1.000,0.907,tie
7,Bank Marketing,MLP+L2,1.000,0.556,tie
8,Bank Marketing,MLP+L2^{(1)},1.000,0.907,tie
9,Bank Marketing,MLP+SDR$_{\rho}$,1.000,0.380,worse



Saved to tables/multi_aso_mcc_opportunity.tex

Fitness Rule: Max(MCC - Eq. Opp.)



...


Saved to tables/multi_aso_acc_parity.tex

Fitness Rule: Max(Acc - Eq. Odds)



,Dataset,Method,ε(Xi→Method),ε(Method→Xi),Result
0,Adult Income,MLP,1.000,1.000,tie
1,Adult Income,MLP+L2,0.730,1.000,tie
2,Adult Income,MLP+L2^{(1)},0.974,1.000,tie
3,Adult Income,MLP+SDR$_{\rho}$,0.674,1.000,tie
4,Adult Income,MLP+SDR$_{\rho_s}$,0.538,1.000,tie
5,Adult Income,MLP+SDR$_{\tau}$,0.649,1.000,tie
6,Bank Marketing,MLP,0.206,1.000,better
7,Bank Marketing,MLP+L2,0.159,1.000,significantly_better
8,Bank Marketing,MLP+L2^{(1)},0.010,1.000,significantly_better
9,Bank Marketing,MLP+SDR$_{\rho}$,0.226,1.000,better



Saved to tables/multi_aso_acc_opportunity.tex

Fitness Rule: Max(Acc - Eq. Opp.)



,Dataset,Method,ε(Xi→Method),ε(Method→Xi),Result
0,Adult Income,MLP,0.338,1.000,better
1,Adult Income,MLP+L2,0.351,1.000,better
2,Adult Income,MLP+L2^{(1)},0.215,1.000,better
3,Adult Income,MLP+SDR$_{\rho}$,0.231,1.000,better
4,Adult Income,MLP+SDR$_{\rho_s}$,1.000,1.000,tie
5,Adult Income,MLP+SDR$_{\tau}$,0.149,1.000,significantly_better
6,Bank Marketing,MLP,1.000,0.366,worse
7,Bank Marketing,MLP+L2,0.582,1.000,tie
8,Bank Marketing,MLP+L2^{(1)},0.380,1.000,better
9,Bank Marketing,MLP+SDR$_{\rho}$,0.832,1.000,tie



Saved to tables/multi_aso_acc_odds.tex


## Tabela Resumo: Contagem de Resultados

In [19]:
# Criar tabela resumo de interpretações
summary_data = []

for fitness_rule in fitness_rules:
    counts = {
        'significantly_better': 0,
        'better': 0,
        'tie': 0,
        'worse': 0,
        'significantly_worse': 0
    }
    
    for entry in multi_aso_data_list:
        if entry['fitness_rule'] == fitness_rule:
            for method, comp_data in entry['comparisons'].items():
                interp = comp_data['interpretation']
                counts[interp] += 1
    
    summary_data.append({
        'Fitness Rule': fitness_rules_abvr[fitness_rule],
        'Sig. Better': counts['significantly_better'],
        'Better': counts['better'],
        'Tie': counts['tie'],
        'Worse': counts['worse'],
        'Sig. Worse': counts['significantly_worse']
    })

summary_df = pd.DataFrame(summary_data)

# Adicionar linha de total
total_row = {
    'Fitness Rule': 'Total',
    'Sig. Better': summary_df['Sig. Better'].sum(),
    'Better': summary_df['Better'].sum(),
    'Tie': summary_df['Tie'].sum(),
    'Worse': summary_df['Worse'].sum(),
    'Sig. Worse': summary_df['Sig. Worse'].sum()
}
summary_df = pd.concat([summary_df, pd.DataFrame([total_row])], ignore_index=True)

print("\nSummary of MLP+SDR$_{\\xi}$ vs Other MLP Methods")
display(summary_df)

summary_df.to_latex('tables/multi_aso_summary.tex', index=False)
print("\nSaved to tables/multi_aso_summary.tex")


Summary of MLP+SDR$_{\xi}$ vs Other MLP Methods


,Fitness Rule,Sig. Better,Better,Tie,Worse,Sig. Worse
0,Max(MCC - Stat. Parity),7,2,12,2,1
1,Max(MCC - Eq. Odds),6,6,10,2,0
2,Max(MCC - Eq. Opp.),2,8,11,3,0
3,Max(Acc - Stat. Parity),2,7,8,6,1
4,Max(Acc - Eq. Odds),5,8,11,0,0
5,Max(Acc - Eq. Opp.),7,5,10,2,0
6,Total,29,36,62,15,2



Saved to tables/multi_aso_summary.tex


## Tabela Resumo: Formato Simbólico (++, +, -, --)

Tabela com símbolos indicando o resultado da comparação MLP+SDR$_{\xi}$ vs cada método:
- `++`: Significativamente melhor (ε < 0.2)
- `+`: Melhor (0.2 ≤ ε < 0.5)
- `≈`: Empate (ε ≥ 0.5 em ambas direções)
- `-`: Pior (0.2 ≤ ε < 0.5)
- `--`: Significativamente pior (ε < 0.2)

In [20]:
# Criar tabela resumo no formato simbólico
def interpretation_to_symbol(interpretation):
    """Converte interpretação para símbolo"""
    symbol_map = {
        'significantly_better': '++',
        'better': '+',
        'tie': '≈',
        'worse': '-',
        'significantly_worse': '--'
    }
    return symbol_map.get(interpretation, '?')

# Preparar dados
methods_compared = [m for m in mlp_methods if m != r'MLP+SDR$_{\xi}$']
rows = []

for dataset in datasets:
    for fitness_rule in fitness_rules:
        row = {
            'Dataset': dataset,
            'Fitness Rule': fitness_rules_abvr[fitness_rule]
        }
        
        # Encontrar entrada correspondente
        for entry in multi_aso_data_list:
            if entry['dataset'] == dataset and entry['fitness_rule'] == fitness_rule:
                for method in methods_compared:
                    if method in entry['comparisons']:
                        symbol = interpretation_to_symbol(entry['comparisons'][method]['interpretation'])
                        row[method] = symbol
                    else:
                        row[method] = '-'
                break
        
        rows.append(row)

symbolic_df = pd.DataFrame(rows)

# Reordenar colunas
cols_order = ['Dataset', 'Fitness Rule'] + methods_compared
symbolic_df = symbolic_df[cols_order]

print("\nSymbolic Summary Table: MLP+SDR$_{\\xi}$ vs Other Methods")
print("Legend: ++ (sig. better), + (better), ≈ (tie), - (worse), -- (sig. worse)\n")
display(symbolic_df)

# Salvar em LaTeX
symbolic_df.to_latex('tables/multi_aso_symbolic_summary.tex', index=False, escape=False)
print("\nSaved to tables/multi_aso_symbolic_summary.tex")


Symbolic Summary Table: MLP+SDR$_{\xi}$ vs Other Methods
Legend: ++ (sig. better), + (better), ≈ (tie), - (worse), -- (sig. worse)



,Dataset,Fitness Rule,MLP,MLP+L2,MLP+L2^{(1)},MLP+SDR$_{\rho}$,MLP+SDR$_{\rho_s}$,MLP+SDR$_{\tau}$
0,Adult Income,Max(MCC - Stat. Parity),≈,≈,--,-,≈,≈
1,Adult Income,Max(MCC - Eq. Odds),+,+,+,+,≈,+
2,Adult Income,Max(MCC - Eq. Opp.),++,++,+,+,+,+
3,Adult Income,Max(Acc - Stat. Parity),≈,+,+,+,+,+
4,Adult Income,Max(Acc - Eq. Odds),≈,≈,≈,≈,≈,≈
5,Adult Income,Max(Acc - Eq. Opp.),+,+,+,+,≈,++
6,Bank Marketing,Max(MCC - Stat. Parity),≈,+,++,≈,≈,≈
7,Bank Marketing,Max(MCC - Eq. Odds),≈,≈,≈,-,≈,≈
8,Bank Marketing,Max(MCC - Eq. Opp.),-,-,+,≈,-,≈
9,Bank Marketing,Max(Acc - Stat. Parity),-,-,++,--,-,-



Saved to tables/multi_aso_symbolic_summary.tex


## Tabela Resumo por Método

Contagem de resultados acumulados por método comparado (todas as fitness rules e datasets).

In [21]:
# Criar tabela resumo por método
method_summary_data = []

# Métodos comparados (excluindo MLP+SDR_xi)
methods_compared = [m for m in mlp_methods if m != r'MLP+SDR$_{\xi}$']

for method in methods_compared:
    counts = {
        'significantly_better': 0,
        'better': 0,
        'tie': 0,
        'worse': 0,
        'significantly_worse': 0
    }
    
    # Acumular contagens para este método em todos os datasets e fitness rules
    for entry in multi_aso_data_list:
        if method in entry['comparisons']:
            interp = entry['comparisons'][method]['interpretation']
            counts[interp] += 1
    
    method_summary_data.append({
        'Method': method,
        'Sig. Better': counts['significantly_better'],
        'Better': counts['better'],
        'Tie': counts['tie'],
        'Worse': counts['worse'],
        'Sig. Worse': counts['significantly_worse']
    })

method_summary_df = pd.DataFrame(method_summary_data)

# Adicionar linha de total
total_row = {
    'Method': 'Total',
    'Sig. Better': method_summary_df['Sig. Better'].sum(),
    'Better': method_summary_df['Better'].sum(),
    'Tie': method_summary_df['Tie'].sum(),
    'Worse': method_summary_df['Worse'].sum(),
    'Sig. Worse': method_summary_df['Sig. Worse'].sum()
}
method_summary_df = pd.concat([method_summary_df, pd.DataFrame([total_row])], ignore_index=True)

print("\nSummary by Method: MLP+SDR$_{\\xi}$ vs Each Method (All Fitness Rules & Datasets)")
display(method_summary_df)

method_summary_df.to_latex('tables/multi_aso_method_summary.tex', index=False, escape=False)
print("\nSaved to tables/multi_aso_method_summary.tex")


Summary by Method: MLP+SDR$_{\xi}$ vs Each Method (All Fitness Rules & Datasets)


,Method,Sig. Better,Better,Tie,Worse,Sig. Worse
0,MLP,5,6,10,3,0
1,MLP+L2,6,6,9,3,0
2,MLP+L2^{(1)},7,7,7,2,1
3,MLP+SDR$_{\rho}$,3,8,9,3,1
4,MLP+SDR$_{\rho_s}$,3,6,12,3,0
5,MLP+SDR$_{\tau}$,5,3,15,1,0
6,Total,29,36,62,15,2



Saved to tables/multi_aso_method_summary.tex
